# Wilcoxon Signed-Rank Test 

In [9]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon, norm

In [10]:
df_raw = pd.read_csv('data_CBR-RAG_2026-04-06_13-14_final.csv', encoding='utf-16', sep='\t')
df_raw = df_raw.copy()
df_raw['participant_id'] = df_raw['CASE']

RAG_COLS = ['Q101','Q104','Q105','Q123','Q109','Q112','Q113','Q116',
            'Q117','Q120','Q121','Q136','Q137','Q140','Q141','Q144','Q145','Q148']

LLM_COLS = ['Q102','Q103','Q106','Q108','Q110','Q111','Q114','Q115',
            'Q118','Q119','Q122','Q135','Q138','Q139','Q142','Q143','Q146','Q147']

DIMENSIONS = {
    '01': 'Helpfulness',
    '02': 'Clarity',
    '03': 'Naturalness',
    '04': 'Length Appropriateness',
    '05': 'Trustworthiness',
    '06': 'Process Grounding'
}

QUESTION_NUMBERS = list(range(1, 19))
print(f'Loaded {len(df_raw)} participants')

Loaded 36 participants


Reverse Scores and Reshape to Long Format                            
Original scale: 1 = Strongly Agree, 5 = Strongly Disagree  
After reversal:  1 = Strongly Disagree, 5 = Strongly Agree  

In [11]:
records = []
for i, q_num in enumerate(QUESTION_NUMBERS):
    for suffix, dim_name in DIMENSIONS.items():
        rag_col = f'{RAG_COLS[i]}_{suffix}'
        llm_col = f'{LLM_COLS[i]}_{suffix}'
        for _, row in df_raw.iterrows():
            if rag_col in df_raw.columns:
                records.append({'participant_id': row['participant_id'], 'question': q_num,
                                 'dimension': dim_name, 'system': 'CBR-RAG',
                                 'score': 6 - row[rag_col]})
            if llm_col in df_raw.columns:
                records.append({'participant_id': row['participant_id'], 'question': q_num,
                                 'dimension': dim_name, 'system': 'LLM',
                                 'score': 6 - row[llm_col]})

df_long = pd.DataFrame(records)
print(f'Total rows: {len(df_long)}')
df_long.head(6)

Total rows: 7776


,participant_id,question,dimension,system,score
0,115,1,Helpfulness,CBR-RAG,2
1,115,1,Helpfulness,LLM,4
2,121,1,Helpfulness,CBR-RAG,3
3,121,1,Helpfulness,LLM,5
4,128,1,Helpfulness,CBR-RAG,5
5,128,1,Helpfulness,LLM,3


## Overall User Score per Participant

In [12]:
rag_user = (df_long[df_long['system'] == 'CBR-RAG']
            .groupby('participant_id')['score'].mean()
            .sort_index())

llm_user = (df_long[df_long['system'] == 'LLM']
            .groupby('participant_id')['score'].mean()
            .sort_index())

user_scores = pd.DataFrame({
    'participant_id': rag_user.index,
    'CBR-RAG user score': rag_user.values.round(4),
    'LLM user score':     llm_user.values.round(4),
    'difference (RAG - LLM)': (rag_user.values - llm_user.values).round(4)
})

print()
print(user_scores.to_string(index=False))


 participant_id  CBR-RAG user score  LLM user score  difference (RAG - LLM)
            115              3.6019          3.5833                  0.0185
            121              3.7037          4.3426                 -0.6389
            128              4.3796          3.4630                  0.9167
            156              4.0741          4.1667                 -0.0926
            157              3.8981          3.2315                  0.6667
            160              3.7963          3.8426                 -0.0463
            164              3.9537          4.2500                 -0.2963
            167              3.6759          3.4537                  0.2222
            168              4.4074          4.2315                  0.1759
            169              3.5093          3.8704                 -0.3611
            178              3.9352          3.4259                  0.5093
            181              4.3519          4.1574                  0.1944
           

## Descriptive Statistics

In [13]:
rag_vals = rag_user.values
llm_vals = llm_user.values

print('Descriptive statistics — overall user score')
print(f'CBR-RAG:  M = {rag_vals.mean():.2f},  SD = {rag_vals.std():.2f},  Mdn = {np.median(rag_vals):.2f}')
print(f'LLM:      M = {llm_vals.mean():.2f},  SD = {llm_vals.std():.2f},  Mdn = {np.median(llm_vals):.2f}')
print()
print(f'Participants where CBR-RAG > LLM: {(rag_vals > llm_vals).sum()}')
print(f'Participants where LLM > CBR-RAG: {(llm_vals > rag_vals).sum()}')
print(f'Participants where CBR-RAG = LLM: {(rag_vals == llm_vals).sum()}')

Descriptive statistics — overall user score
CBR-RAG:  M = 3.86,  SD = 0.32,  Mdn = 3.80
LLM:      M = 3.75,  SD = 0.39,  Mdn = 3.69

Participants where CBR-RAG > LLM: 21
Participants where LLM > CBR-RAG: 15
Participants where CBR-RAG = LLM: 0


## One-Sided Wilcoxon Signed-Rank Test

In [14]:
W, p = wilcoxon(rag_vals, llm_vals, alternative='greater')

z = norm.ppf(1 - p)
r = z / np.sqrt(len(rag_vals))

if p < 0.001:
    sig = '***'
elif p < 0.01:
    sig = '**'
elif p < 0.05:
    sig = '*'
else:
    sig = 'ns'

print('Wilcoxon Signed-Rank Test — Overall User Score')
print(f'W        = {W}')
print(f'p-value  = {p:.4f}  {sig}')
print(f'r        = {r:.3f}')
print()

Wilcoxon Signed-Rank Test — Overall User Score
W        = 448.5
p-value  = 0.0348  *
r        = 0.302



In [15]:
user_scores.to_csv('user_scores.csv', index=False)

summary = pd.DataFrame([{
    'M (CBR-RAG)': round(rag_vals.mean(), 2),
    'M (LLM)':     round(llm_vals.mean(), 2),
    'W':            W,
    'p-value':      round(p, 4),
    'r':            round(r, 3),
    'Sig.':         sig,
}])
summary.to_csv('wilcoxon_user_score_results.csv', index=False)

In [16]:
print('Overall User Score Results')
print(f'{"":25} {"CBR-RAG":>10} {"LLM":>10}')
print(f'{"Mean user score":25} {rag_vals.mean():>10.2f} {llm_vals.mean():>10.2f}')
print(f'{"SD":25} {rag_vals.std():>10.2f} {llm_vals.std():>10.2f}')
print(f'{"Median":25} {np.median(rag_vals):>10.2f} {np.median(llm_vals):>10.2f}')
print(f'{"Min":25} {rag_vals.min():>10.2f} {llm_vals.min():>10.2f}')
print(f'{"Max":25} {rag_vals.max():>10.2f} {llm_vals.max():>10.2f}')
print()
print(f'{"Wilcoxon W":25} {W:>10.1f}')
print(f'{"p-value":25} {p:>10.4f}')
print(f'{"Effect size r":25} {r:>10.3f}')
print(f'{"Significance":25} {sig:>10}')
print()
print(f'Participants CBR-RAG > LLM: {(rag_vals > llm_vals).sum()}')
print(f'Participants LLM > CBR-RAG: {(llm_vals > rag_vals).sum()}')
print()
print('Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')
print('Effect size r: small>=0.1  medium>=0.3  large>=0.5')

Overall User Score Results
                             CBR-RAG        LLM
Mean user score                 3.86       3.75
SD                              0.32       0.39
Median                          3.80       3.69
Min                             3.30       3.18
Max                             4.55       4.73

Wilcoxon W                     448.5
p-value                       0.0348
Effect size r                  0.302
Significance                       *

Participants CBR-RAG > LLM: 21
Participants LLM > CBR-RAG: 15

Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant
Effect size r: small>=0.1  medium>=0.3  large>=0.5
